In [1]:
def early_finish_time(s, f):
    selected = []
    last_finish = -1

    for i in range(len(s)):  # already sorted by finish time in your question
        if s[i] >= last_finish:
            selected.append(i + 1)     # store activity number (1-based)
            last_finish = f[i]

    return selected


# Test
s = [1, 3, 0, 5, 3, 5, 6, 8, 8, 2, 12]
f = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

print("Early Finish Time selected activities:", early_finish_time(s, f))

Early Finish Time selected activities: [1, 4, 8, 11]


In [2]:
def shortest_interval_first(s, f):
    activities = []
    for i in range(len(s)):
        duration = f[i] - s[i]
        activities.append((duration, f[i], s[i], i + 1))  # (dur, finish, start, id)

    activities.sort()  # sort by duration (then finish)

    selected = []
    chosen_intervals = []

    for dur, finish, start, act_id in activities:
        overlap = False
        for (cs, cf) in chosen_intervals:
            # overlap if NOT (finish <= cs OR start >= cf)
            if not (finish <= cs or start >= cf):
                overlap = True
                break

        if not overlap:
            selected.append(act_id)
            chosen_intervals.append((start, finish))

    # optional: sort selected activities by finish time for nice display
    selected.sort(key=lambda idx: f[idx - 1])
    return selected


print("Shortest Interval First selected activities:", shortest_interval_first(s, f))

Shortest Interval First selected activities: [2, 4, 8, 11]


In [ ]:
# ---------- Activity Selection using Greedy ----------

# Method 1: Early Finish Time (Optimal)
def early_finish(activities):
    # sort by finish time
    acts = sorted(enumerate(activities, start=1), key=lambda x: x[1][1])

    selected = []
    last_finish = -1

    for idx, (start, finish) in acts:
        if start >= last_finish:
            selected.append(idx)
            last_finish = finish

    return selected


# Method 2: Shortest Interval First (Heuristic)
def shortest_interval(activities):
    # sort by duration
    acts = sorted(
        enumerate(activities, start=1),
        key=lambda x: (x[1][1] - x[1][0])
    )

    selected = []
    last_finish = -1

    for idx, (start, finish) in acts:
        if start >= last_finish:
            selected.append(idx)
            last_finish = finish

    return selected


# Helper function to print results nicely
def print_selected(name, selected, activities):
    print(name)
    for i in selected:
        print(f"Activity {i}: {activities[i-1]}")
    print()


# ---------- Main Program ----------
activities = [
    (1, 4),
    (3, 5),
    (0, 6),
    (5, 7),
    (3, 8),
    (5, 9),
    (6, 10),
    (8, 11),
    (8, 12),
    (2, 13),
    (12, 14)
]

# Run both methods
sel1 = early_finish(activities)
sel2 = shortest_interval(activities)

# Print results
print_selected("Early Finish Time:", sel1, activities)
print_selected("Shortest Interval First:", sel2, activities)

In this program we solve the Activity Selection problem using two greedy ideas. The input is activities, a list of tuples like (start, finish), where each tuple represents one activity’s start time and finish time. We always keep the original activity numbers (1, 2, 3, …) so we can report “Activity 4” etc. That’s why both methods use enumerate(activities, start=1): it attaches an index idx (activity number) to each (start, finish) pair.

Method 1: early_finish(activities) (optimal greedy)

Inside early_finish, we create acts, which is the list of activities with their index, sorted by finish time. In the code, x[1][1] means: from each item x = (idx, (start, finish)), take x[1] (the (start, finish) tuple) and then x[1] again (the finish). After sorting, we start selecting activities. We keep a list called selected to store the chosen activity numbers. We also keep last_finish, which stores the finish time of the most recently selected activity. It starts at -1 so the first compatible activity will always be selected. Then we loop through acts using for idx, (start, finish) in acts:. For each activity, we check if start >= last_finish: meaning “this activity starts after the last chosen one ends, so it doesn’t overlap.” If it’s compatible, we add its idx to selected and update last_finish = finish so future activities must start after this new finish time. At the end, the function returns selected.

Method 2: shortest_interval(activities) (simple greedy heuristic)

This method is similar, but instead of sorting by finish time, it sorts by duration. Duration means how long an activity runs: finish - start. So we again build acts using enumerate, but now the sorting key is x[1][1] - x[1][0] (finish minus start). That gives shortest activities first. Then we do the same selection process: selected stores chosen activity numbers, and last_finish stores the finish time of the most recent chosen activity. We loop through each (start, finish) in duration order, pick it if start >= last_finish, and update last_finish to the current finish. This approach is easy to understand, but it’s not guaranteed to be optimal for every dataset (unlike early-finish, which is the standard optimal greedy for maximizing the number of activities).

Printing the result: print_selected(name, selected, activities)

The helper function print_selected just displays output neatly. name is a label like "Early Finish Time:". selected is the list of selected activity numbers returned by a method. activities is the original list so we can print the real (start, finish) times. The loop for i in selected: prints each chosen activity number and its time interval using activities[i-1] (because activity numbers start at 1 but Python lists start at index 0).

Main flow

In the main part, we define the activities list from your table, run both greedy methods (sel1 = early_finish(activities) and sel2 = shortest_interval(activities)), and then print both results. So overall: the program sorts activities according to a greedy rule, uses last_finish to ensure no overlap, stores chosen activity numbers in selected, and prints them clearly along with their start/finish times.

If you want, I can also give you a short 5–6 line viva answer to memorize for “explain the algorithm” and “explain variables.”

In [ ]:
import heapq
from collections import defaultdict

class Node:
    def __init__(self, freq, symbol=None, left=None, right=None):
        self.freq = freq
        self.symbol = symbol
        self.left = left
        self.right = right

    # needed for heapq comparisons
    def __lt__(self, other):
        return self.freq < other.freq


def build_huffman_codes(freqs):
    # create priority queue
    heap = [Node(freq, sym) for sym, freq in freqs.items()]
    heapq.heapify(heap)

    # greedy merges
    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        parent = Node(left.freq + right.freq, left=left, right=right)
        heapq.heappush(heap, parent)

    # traverse tree to get codes
    root = heap[0]
    codes = {}

    def dfs(node, code):
        if node.symbol is not None:
            codes[node.symbol] = code
            return
        dfs(node.left, code + "0")
        dfs(node.right, code + "1")

    dfs(root, "")
    return codes


# ===== Example (your data) =====
freqs = {
    'f': 5,
    'e': 9,
    'c': 12,
    'b': 13,
    'd': 16,
    'a': 45
}

codes = build_huffman_codes(freqs)

print("Huffman Codes:")
for k in sorted(codes):
    print(f"{k}: {codes[k]}")